In [ ]:
import csv
import traceback

def csv_to_longtable(
    csv_path,
    output_path,
    caption,
    label,
    sort_by=None,
    descending=False,
    num_summary_rows=0,
    round_digits=None
):
    with open(csv_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        rows = list(reader)

    header = rows[0]
    data_rows = rows[1:]

    expected_len = len(header)
    filtered_rows = []
    for i, row in enumerate(data_rows):
        if len(row) == expected_len:
            filtered_rows.append(row)
        else:
            print(f"Warning: Sor {i+2} kihagyva, mert hibás hossza van: {len(row)} (várt: {expected_len})")

    data_rows = filtered_rows

    if num_summary_rows > 0:
        normal_rows = data_rows[:-num_summary_rows]
        summary_rows = data_rows[-num_summary_rows:]
    else:
        normal_rows = data_rows
        summary_rows = []

    # Rendezés, ha kell
    if sort_by is not None:
        try:
            sort_index = header.index(sort_by)
            try:
                normal_rows.sort(
                    key=lambda x: float(x[sort_index]),
                    reverse=descending
                )
            except Exception:
                print(">>> HIBA TÖRTÉNT A RENDEZÉSNÉL:")
                traceback.print_exc()
                print(">>> Hibás sorok float-konverzió ellenőrzése:")
                for i, row in enumerate(normal_rows):
                    try:
                        float(row[sort_index])
                    except Exception as e_row:
                        print(f"Sor {i+2}: '{row[sort_index]}' - {type(e_row).__name__}: {e_row}")
                print(">>> Rendezés sikertelen, eredeti sorrendet használom.")
        except (ValueError, IndexError):
            print(f"Warning: Nem találtam meg az oszlopot ('{sort_by}').")

    final_rows = normal_rows + summary_rows

    num_columns = len(header)
    column_format = "l" * num_columns  # Minden balra zárt

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("\\begin{center}\n")
        f.write("\\begin{longtable}{" + column_format + "}\n")
        f.write("\\caption{" + caption + "}\\label{" + label + "}\\\\\n")
        f.write("\\toprule\n")
        f.write(" & ".join(header) + " \\\\\n")
        f.write("\\midrule\n")
        f.write("\\endfirsthead\n")

        f.write("\\toprule\n")
        f.write(" & ".join(header) + " \\\\\n")
        f.write("\\midrule\n")
        f.write("\\endhead\n")

        for idx, row in enumerate(final_rows):
            if num_summary_rows > 0 and idx == len(normal_rows):
                f.write("\\midrule\n")

            sanitized_row = []
            for cell in row:
                cell = cell.replace('_', '-')  # Underscore helyett kötőjel
                if round_digits is not None:
                    try:
                        number = float(cell)
                        cell = f"{number:.{round_digits}f}"
                    except ValueError:
                        pass  # Nem szám, nem formázzuk
                sanitized_row.append(cell)

            f.write(" & ".join(sanitized_row) + " \\\\\n")

        f.write("\\bottomrule\n")
        f.write("\\end{longtable}\n")
        f.write("\\end{center}\n")

# --- Példa hívás:
csv_to_longtable(
    csv_path="data.csv",
    output_path="phase3_NASNet_summary_table.tex",
    caption="Összehasonlító táblázat a NASNet iteratív tanításáról.",
    label="tab:phase3_NASNet_summary_table",
    sort_by="xxx",
    descending=True,
    num_summary_rows=0,
    round_digits=None  # ÚJDONSÁG: 2 tizedesre vágunk
)
